# Registration and Metrics for full images

In [ ]:
import sys
sys.path.insert(0, "/home-link/zxovq55/Masters-Thesis/src/registration")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tifffile import imread
from skimage.transform import resize
import numpy as np

import importlib, registration
importlib.reload(registration)
importlib.reload(registration.reg)
importlib.reload(registration.metrics)
importlib.reload(registration.regPipeline)
importlib.reload(registration.preprocess)

## Registration with regPipeline

#### H&E as moving image

In [ ]:
def overlay_images(dapi_img, hne_img, title='overlay'):
    h, w = dapi_img.shape
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)
    plt.title(title)
    plt.axis([0, w, h, 0])
    
    legend_patches = [
        Patch(color='green', label='DAPI', alpha=0.8),
        Patch(color='red', label='HnE', alpha=0.4)
    ]
    plt.legend(handles=legend_patches)

def display_img_comparison(dapi_img, hne_img, hne_deconv, registered_hne_imgs):
    no_comparisons = len(registered_hne_imgs)
    if 'intensity based' in registered_hne_imgs:
        no_comparisons += len(registered_hne_imgs['intensity based']) - 1

    h, w = dapi_img.shape
    
    plt.figure(figsize=((no_comparisons+3)*3, 6))

    plt.subplot(1, no_comparisons + 3, 1)
    plt.title('DAPI image')
    plt.imshow(dapi_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 2)
    plt.title('HnE image')
    plt.imshow(hne_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 3)
    plt.title('Initial Overlay')
    overlay_images(dapi_img, hne_deconv, title='initial overlay')

    for key, value in registered_hne_imgs.items():
        if key == 'intensity based':
            for key1, value1 in value.items():
                plt.subplot(1, no_comparisons + 3, list(value.keys()).index(key1) + 5)
                overlay_images(dapi_img, value1, title=key1)
        else:
            plt.subplot(1, no_comparisons + 3, list(registered_hne_imgs.keys()).index(key) + 4)
            overlay_images(dapi_img, value, title=key)

    plt.tight_layout()
    plt.show()

In [ ]:
hne_px_sz = 0.5023
dapi_px_sz = 0.209877

scale = hne_px_sz / dapi_px_sz

# load dapi
dapi_1_data = np.array(imread("../../../data/dapi_1.tif"))
dapi1_init = resize(dapi_1_data, (int(dapi_1_data.shape[0]/scale), int(dapi_1_data.shape[1]/scale)), anti_aliasing=True)

dapi_2_data = np.array(imread("../../../data/dapi_2.tif"))
dapi2_init = resize(dapi_2_data, (int(dapi_2_data.shape[0]/scale), int(dapi_2_data.shape[1]/scale)), anti_aliasing=True)

dapi_3_data = np.array(imread("../../../data/dapi_3.tif"))
dapi3_init = resize(dapi_3_data, (int(dapi_3_data.shape[0]/scale), int(dapi_3_data.shape[1]/scale)), anti_aliasing=True)

dapi1_init, dapi2_init, dapi3_init = dapi1_init*255, dapi2_init*255, dapi3_init*255

# load hne
hne_1_init = np.array(imread("../../../data/hne_1.tif"))
hne_2_init = np.array(imread("../../../data/hne_2.tif"))
hne_3_init = np.array(imread("../../../data/hne_3.tif"))


In [ ]:
# preprocess hne
hne1_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_1_init)
hne2_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_2_init)
hne3_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_3_init)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/dapi_1.tif", 
                                                                                   "../../../data/hne_1.tif", 0.209877, 0.5023, fixed_img='multiplexed')
display_img_comparison(dapi1_init, hne_1_init, hne1_deconv, registered_imgs)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/dapi_2.tif", 
                                                                                   "../../../data/hne_2.tif", 0.209877, 0.5023, fixed_img='multiplexed')
display_img_comparison(dapi2_init, hne_2_init, hne2_deconv, registered_imgs)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/dapi_3.tif", 
                                                                                   "../../../data/hne_3.tif", 0.209877, 0.5023, fixed_img='multiplexed')
display_img_comparison(dapi3_init, hne_3_init, hne3_deconv, registered_imgs)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/dapi_1.tif", 
                                                                                   "../../../data/hne_1.tif", 0.209877, 0.5023, fixed_img='multiplexed',
                                                                                   adv_tform='intensity', intensity_tform='af-bs')
display_img_comparison(dapi1_init, hne_1_init, hne1_deconv, registered_imgs)

In [ ]:
# registration with full channel multiplex image (extracting DAPI channel)
img_raw = imread("../../../data/rack-01-well-A01-roi-002-exp-2_backsub.ome.tif") 
img = np.array(img_raw).transpose(1,2,0)
img_ch = registration.extract_channel(img, 0)
img_ch_scaled = resize(img_ch, (int(img_ch.shape[0]/scale), int(img_ch.shape[1]/scale)), anti_aliasing=True)
img_ch_scaled = img_ch_scaled*255

transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/rack-01-well-A01-roi-002-exp-2_backsub.ome.tif", 
                                                                                   "../../../data/hne_2.tif", 0.209877, 0.5023, fixed_img='multiplexed',
                                                                                   adv_tform='intensity', intensity_tform='af-bs')
display_img_comparison(img_ch_scaled, hne_2_init, hne2_deconv, registered_imgs)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/dapi_2.tif", 
                                                                                   "../../../data/hne_2.tif", 0.209877, 0.5023, fixed_img='multiplexed',
                                                                                   adv_tform='intensity', intensity_tform='r-af-bs')
display_img_comparison(dapi2_init, hne_2_init, hne2_deconv, registered_imgs)

#### DAPI as moving image

In [ ]:
def overlay_images_hnef(hne_img, dapi_img, hne_deconv, title='overlay'):
    h, w = hne_deconv.shape
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)
    plt.title(title)
    plt.axis([0, w, h, 0])
    
    legend_patches = [
        Patch(color='green', label='DAPI', alpha=0.8),
        Patch(color='red', label='HnE', alpha=0.4)
    ]
    plt.legend(handles=legend_patches)

def display_img_comparison_hnef(dapi_img, hne_img, hne_deconv, registered_dapi_imgs):
    no_comparisons = len(registered_dapi_imgs)
    if 'intensity based' in registered_dapi_imgs:
        no_comparisons += len(registered_dapi_imgs['intensity based']) - 1

    h, w = hne_deconv.shape
    
    plt.figure(figsize=((no_comparisons+3)*3, 6))

    plt.subplot(1, no_comparisons + 3, 1)
    img = hne_img.astype(float)
    img = img / img.max()
    plt.title('HnE image')
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 2)
    plt.title('DAPI image')
    plt.imshow(dapi_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 3)
    plt.title('Initial Overlay')
    overlay_images_hnef(hne_deconv, dapi_img, hne_deconv, title='initial overlay')

    for key, value in registered_dapi_imgs.items():
        if key == 'intensity based':
            for key1, value1 in value.items():
                plt.subplot(1, no_comparisons + 3, list(value.keys()).index(key1) + 5)
                overlay_images_hnef(hne_deconv, value1, hne_deconv, title=key1)
        else:
            plt.subplot(1, no_comparisons + 3, list(registered_dapi_imgs.keys()).index(key) + 4)
            overlay_images_hnef(hne_deconv, value, hne_deconv, title=key)

    plt.tight_layout()
    plt.show()

In [ ]:
hne_px_sz = 0.5023
dapi_px_sz = 0.209877

scale = hne_px_sz / dapi_px_sz

# load dapi
dapi_1_init = np.array(imread("../../../data/dapi_1.tif"))
dapi_2_init = np.array(imread("../../../data/dapi_2.tif"))
dapi_3_init = np.array(imread("../../../data/dapi_3.tif"))

# load hne
hne_1_data = np.array(imread("../../../data/hne_1.tif"))
hne_1_init = resize(hne_1_data, (int(hne_1_data.shape[0]*scale), int(hne_1_data.shape[1]*scale)), anti_aliasing=True)

hne_2_data = np.array(imread("../../../data/hne_2.tif"))
hne_2_init = resize(hne_2_data, (int(hne_2_data.shape[0]*scale), int(hne_2_data.shape[1]*scale)), anti_aliasing=True)

hne_3_data = np.array(imread("../../../data/hne_3.tif"))
hne_3_init = resize(hne_3_data, (int(hne_3_data.shape[0]*scale), int(hne_3_data.shape[1]*scale)), anti_aliasing=True)

hne_1_init, hne_2_init, hne_3_init = hne_1_init*255, hne_2_init*255, hne_3_init*255

In [ ]:
# preprocess hne
hne_1_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_1_init)
hne_2_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_2_init)
hne_3_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_3_init)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/hne_1.tif", 
                                                                                   "../../../data/dapi_1.tif", 0.5023, 0.209877, fixed_img='hne')
display_img_comparison_hnef(dapi_1_init, hne_1_init, hne_1_deconv, registered_imgs)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/hne_2.tif", 
                                                                                   "../../../data/dapi_2.tif", 0.5023, 0.209877, fixed_img='hne')
display_img_comparison_hnef(dapi_2_init, hne_2_init, hne_2_deconv, registered_imgs)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/hne_3.tif", 
                                                                                   "../../../data/dapi_3.tif", 0.5023, 0.209877, fixed_img='hne')
display_img_comparison_hnef(dapi_3_init, hne_3_init, hne_3_deconv, registered_imgs)

In [ ]:
transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/hne_1.tif", 
                                                                                   "../../../data/dapi_1.tif", 0.5023, 0.209877, fixed_img='hne', adv_tform='intensity', intensity_tform='bspline')
display_img_comparison_hnef(dapi_1_init, hne_1_init, hne_1_deconv, registered_imgs)

In [ ]:
# registration with full channel multiplex image (extracting DAPI channel)
img_raw = imread("../../../data/rack-01-well-A01-roi-002-exp-2_backsub.ome.tif") 
img = np.array(img_raw).transpose(1,2,0)
img_ch = registration.extract_channel(img, 0)

transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/hne_2.tif", 
                                                                                   "../../../data/rack-01-well-A01-roi-002-exp-2_backsub.ome.tif", 0.5023, 0.209877, fixed_img='hne')
display_img_comparison_hnef(img_ch, hne_2_init, hne_2_deconv, registered_imgs)

In [ ]:
# registration with full channel multiplex image (extracting DAPI channel)
img_raw = imread("../../../data/rack-01-well-A01-roi-002-exp-2_backsub.ome.tif") 
img = np.array(img_raw).transpose(1,2,0)
img_ch = registration.extract_channel(img, 0)

transformation_maps, registered_imgs, moved_img, tre, mi = registration.registration_pipeline("../../../data/hne_2.tif", 
                                                                                   "../../../data/rack-01-well-A01-roi-002-exp-2_backsub.ome.tif", 0.5023, 0.209877, fixed_img='hne', adv_tform='intensity', intensity_tform='bspline')
display_img_comparison_hnef(img_ch, hne_2_init, hne_2_deconv, registered_imgs)